# Notebook 2 — Locked alias strategy and model comparison

This notebook consumes the 15 completed runs from `04_five_model_comparison.ipynb`. It selects one global alias strategy using mean graded nDCG, with mean MAP as the tie-breaker, then reports the five-model comparison under that locked strategy. No models are downloaded or re-encoded.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
if not (ROOT / "ranking_metrics.py").exists():
    raise FileNotFoundError(f"Repository root not found from {Path.cwd()}")

RESULTS_DIR = ROOT / "results"
COMPARISON_PATH = RESULTS_DIR / "model_alias_comparison_15.csv"
if not COMPARISON_PATH.exists():
    raise FileNotFoundError(f"Missing {COMPARISON_PATH}; run Notebook 1 first")

comparison = pd.read_csv(COMPARISON_PATH)
expected_models = {"minilm", "mpnet", "jobbert", "bge", "e5"}
expected_aliases = {"first", "mean", "max"}
required = {"model_short_name", "model_name", "skill_representation", "ndcg_graded", "map_binary", "mrr_binary", "p@1", "p@5", "p@10"}
if len(comparison) != 15 or set(comparison["model_short_name"]) != expected_models or set(comparison["skill_representation"]) != expected_aliases:
    raise ValueError("Expected exactly 15 results for five models and three alias strategies")
if required.difference(comparison.columns):
    raise ValueError(f"Missing columns: {sorted(required.difference(comparison.columns))}")
if not comparison.groupby("skill_representation")["model_short_name"].nunique().eq(5).all():
    raise ValueError("Each alias strategy must have one result for every model")
print(f"Loaded {len(comparison)} configurations")

Loaded 15 configurations


## Select the global alias strategy

The primary metric is mean graded nDCG across models. Mean binary MAP is used only as a tie-breaker.

In [2]:
alias_summary = (
    comparison.groupby("skill_representation", as_index=False)
    .agg(mean_ndcg_graded=("ndcg_graded", "mean"), mean_map_binary=("map_binary", "mean"), mean_mrr_binary=("mrr_binary", "mean"))
    .sort_values(["mean_ndcg_graded", "mean_map_binary"], ascending=False)
    .reset_index(drop=True)
)
selected_alias = str(alias_summary.iloc[0]["skill_representation"])
alias_summary["selected"] = alias_summary["skill_representation"].eq(selected_alias)
selection = {
    "selected_alias_strategy": selected_alias,
    "selection_metric": "mean_ndcg_graded",
    "tie_breaker": "mean_map_binary",
    "mean_ndcg_graded": float(alias_summary.iloc[0]["mean_ndcg_graded"]),
    "mean_map_binary": float(alias_summary.iloc[0]["mean_map_binary"]),
}
display(alias_summary)
alias_summary.to_csv(RESULTS_DIR / "locked_alias_summary.csv", index=False)
(RESULTS_DIR / "locked_alias_selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
print(f"Selected alias strategy: {selected_alias}")

,skill_representation,mean_ndcg_graded,mean_map_binary,mean_mrr_binary,selected
0,mean,0.652597,0.194036,0.676697,True
1,first,0.650188,0.195984,0.683031,False
2,max,0.642592,0.180798,0.647566,False


Selected alias strategy: mean


## Official locked model comparison

In [3]:
locked = (
    comparison[comparison["skill_representation"] == selected_alias]
    .sort_values(["ndcg_graded", "map_binary"], ascending=False)
    .reset_index(drop=True)
)
if len(locked) != 5:
    raise ValueError(f"Expected five locked models, found {len(locked)}")
locked.insert(0, "rank", range(1, 6))
display(locked[["rank", "model_short_name", "model_name", "skill_representation", "ndcg_graded", "ndcg_binary", "map_binary", "mrr_binary", "p@1", "p@5", "p@10"]])
locked.to_csv(RESULTS_DIR / "locked_model_comparison.csv", index=False)
print("Saved: results/locked_model_comparison.csv")

,rank,model_short_name,model_name,skill_representation,ndcg_graded,ndcg_binary,map_binary,mrr_binary,p@1,p@5,p@10
0,1,jobbert,TechWolf/JobBERT-v2,mean,0.738077,0.788888,0.341614,0.788649,0.648026,0.562500,0.517763
1,2,e5,intfloat/e5-base-v2,mean,0.640110,0.682140,0.164249,0.695816,0.542763,0.450658,0.416776
2,3,mpnet,all-mpnet-base-v2,mean,0.632662,0.676252,0.161164,0.630215,0.453947,0.398026,0.371053
3,4,bge,BAAI/bge-base-en-v1.5,mean,0.630855,0.672003,0.155516,0.634001,0.444079,0.413816,0.373684
4,5,minilm,all-MiniLM-L6-v2,mean,0.621283,0.664406,0.147637,0.634803,0.470395,0.383553,0.365461


Saved: results/locked_model_comparison.csv


## Select the three fine-tuning candidates

The top three models under the locked strategy are exported for Notebook 3.

In [4]:
selected_models = locked.head(3)[["rank", "model_short_name", "model_name"]].copy()
selected_models["alias_strategy"] = selected_alias
selected_models.to_csv(RESULTS_DIR / "selected_models_for_finetuning.csv", index=False)
display(selected_models)
print("Saved: results/selected_models_for_finetuning.csv")

,rank,model_short_name,model_name,alias_strategy
0,1,jobbert,TechWolf/JobBERT-v2,mean
1,2,e5,intfloat/e5-base-v2,mean
2,3,mpnet,all-mpnet-base-v2,mean


Saved: results/selected_models_for_finetuning.csv
